**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/semenov-2026-001521"], check=True)
    os.chdir("/content/encore/entries/semenov-2026-001521")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Semenov et al. (2026) from dandiset 001521

Paper: Semenov I, Silkuniene G, Silkunas M, Pakhomova ON, Bixler A, Ibey BL, Kochunov P, Pakhomov AG
(2026). Sub-microsecond optical measurements of cell membrane charging and lesioning by pulsed electric
fields. Bioelectrochemistry 171, 109283.
[doi:10.1016/j.bioelechem.2026.109283](https://doi.org/10.1016/j.bioelechem.2026.109283). The claims
file was written from the version of record (main text with Materials and methods, figure legends and
Data availability; the paper has no supplement and links no code).

Dandiset: [DANDI:001521](https://dandiarchive.org/dandiset/001521/0.261006.1845), version
0.261006.1845. The paper cites no version of the dandiset (its Data availability says data are
available on request). This is the only published version, so it is pinned; the draft lists the same 73
assets with the same total size.

What the dandiset holds: 73 widefield Fluo-4 time-lapses of CHO cells (one NWB file each, no ROIs or
traces), in two sessions. The 2025-02-03 session (31 files) is the single 1-us pulse experiment of
Fig. 6A, and the 2025-05-02 session (42 files) is the 60-pulse 6 Hz train experiment of Fig. 6B. The
pulsed-laser potentiometric imaging behind Figs. 1 to 5 and 7 is not in it. Six other dandisets of the
same lab name the same paper (001515, 001516, 001517, 001518, 001525 and 001526); by their titles they
hold the potentiometric experiments, and they are targets of their own.

Each file gives the pulse as a generator voltage, while the paper gives only field strengths. Two
readings of the conversion are run throughout, and the one that fits the fields and group sizes of Fig. 6B
is adopted (see "Field strengths"). The cell regions are found here
by a standard segmentation, as the paper drew them by hand in MetaMorph (see `fluo4.py`).

No random numbers are drawn anywhere in this notebook.

| Claim | Status |
|---|---|
| fig6a-ca-entry-1000-1200 | tested |
| fig6a-ca-sustained | tested |
| fig6a-780-not-significant | tested |
| fig6b-train-threshold | tested |
| fig6b-650-weak-but-significant | tested |
| fig5b-conductivity-calibration | tested (a computation from Eq. 2, no recordings) |
| fig1-pole-leakage-pattern, fig2a-fluovolt-saturation, fig2a-fluovolt-stable-within-set, fig2b-di8-decline-within-set, fig2cd-stable-then-drop, fig2c-1000-within-set-trend, fig3-dye-calibration, fig3-fluovolt-nonlinear, fig4a-fluovolt-maximum, fig4b-di8-maximum, fig4b-first-10-pulses, fig4b-cathodic-asymmetry, fig4c-equator-cosine, fig4d-short-pulses-same-limit, fig5a-gd-extends-anodic, fig5a-gd-cathodic-not-higher, fig5c-gd-reduces-conductivity, fig7a-test-pulse-cosine, fig7b-conditioning-sublinear, fig7b-test-pulse-recovery, fig7c-fast-resealing | blocked: data missing (potentiometric imaging not in this dandiset) |


In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import fluo4

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
print("dandiset", fluo4.DANDISET_ID, "version", fluo4.DANDISET_VERSION)

In [ ]:
t0 = time.time()
files = fluo4.list_files()
res = fluo4.process_all(files.url.tolist())
print(f"{len(res)} files streamed and reduced to cell traces in {time.time() - t0:.0f} s")

## Data check

Every file is streamed once and reduced, as it loads, to one background-corrected trace per cell (about
four minutes). Each check of the metadata against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{
    "path": p, "set": r["set"], "session_id": r["session_id"], "subject_id": r["subject_id"],
    "session_start": r["session_start"], "volts": r["volts"], "n_pulses": r["n_pulses"],
    "gap_mm": r["gap_mm"], "rate_hz": r["rate_hz"], "frames": r["shape"][0], "height": r["shape"][1],
    "width": r["shape"][2], "duration_s": r["shape"][0] / r["rate_hz"], "pulse_frame": r["pulse_frame"],
    "unit": r["unit"], "dtype": r["dtype"], "raw_min": r["raw_min"], "raw_max": r["raw_max"],
    "cells": r["dF"].shape[0]} for p, r in zip(files.path, res)])
info["fov"] = info.subject_id.str.replace("CHO_", "")
print("Files by set (session) and voltage:")
print(pd.crosstab(info.set, info.volts, margins=True).to_string())
print("\nFiles by set and pulses per file (0 = sham):")
print(pd.crosstab(info.set, info.n_pulses, margins=True).to_string())
print("\nSubject age and species as the files give them:",
      sorted({(r["subject_age"], r["subject_species"]) for r in res}))
print("\nPer set: frames, rate, duration, pulse frame, image size, electrode gap, unit, dtype, raw range")
print(info.groupby("set").agg(files=("path", "size"), subjects=("subject_id", "nunique"),
                              frames=("frames", "unique"), rate_hz=("rate_hz", "unique"),
                              duration_s=("duration_s", "unique"), pulse_frame=("pulse_frame", "unique"),
                              height=("height", "unique"), width=("width", "unique"),
                              gap_mm=("gap_mm", "unique"), unit=("unit", "unique"), dtype=("dtype", "unique"),
                              raw_min=("raw_min", "min"), raw_max=("raw_max", "max")).T.to_string())
print("\nGroups and objects held by the files (union over files):")
for key in ["groups", "acquisition", "processing", "stimulus", "analysis"]:
    print(f"  {key}:", sorted({x for r in res for x in r[key]}))
print("\nImage series descriptions:")
for s in sorted({r["series_description"] for r in res}):
    print("  -", s)
print("\nExample session_description (first file of each set):")
for s in ["single", "train"]:
    print(f"  [{s}]", next(r["description"] for r in res if r["set"] == s)[:400], "...")

The files hold one image series each and nothing else (no ROIs, traces, stimulus series or
tables). The image unit is "n.a."; the values are raw camera counts (minimum 119, above the camera
offset). The rates, frame counts and durations match the series descriptions and Methods 2.4: single
pulses at 1 frame/s for 300 s with the pulse at 60 s; trains at 0.5 frame/s for 180 s with the train
from 30 s (Methods: "a burst of 60 pulses at 6 Hz began at 30 s ... and recording continued for 150 s",
30 + 150 = 180 s). Every file states an electrode gap of 1.00 mm; the paper's Methods 2.2 give a 1.6 mm
spacing. The dandiset's title and description mention only single pulses (next cell), but 33 of the 73 files are
trains of 60 pulses; the other 9 files of the train session are shams. Every subject has an age of P1D,
though subjects are fields of view of a cell line, and the species NCBITaxon 10029, which the archive's
summary names "Cricetulus griseus - Cricetulus aureus", two species for one taxon (next cell).

The archive's metadata, for this dandiset and for the six other dandisets of the lab that name the same paper
(each lists the paper's DOI as `IsPublishedIn`). By their titles the six hold the membrane polarization and
recovery experiments; they are targets of their own and are not used here.

In [ ]:
# Archive metadata (DANDI API) at each dandiset's most recent published version.
from dandi.dandiapi import DandiAPIClient
client = DandiAPIClient()
meta = client.get_dandiset(fluo4.DANDISET_ID, fluo4.DANDISET_VERSION).get_raw_metadata()
print("001521 description:", meta["description"][:420], "...")
print("001521: species", [(s["name"], s["identifier"]) for s in meta["assetsSummary"].get("species", [])])
for d in ["001521", "001515", "001516", "001517", "001518", "001525", "001526"]:
    ds = client.get_dandiset(d)
    m = ds.get_raw_metadata()
    print(f"{d} {ds.version_id}: {m['name']}")
    print("    related:", [(r.get("relation"), r.get("identifier")) for r in m.get("relatedResource", [])])

In [ ]:
# Is a "subject" one cell, one dish or one field of view? Correlate the pre-pulse mean images.
Z = np.array([(r["pre_small"] - r["pre_small"].mean()).ravel() for r in res])
Z /= np.linalg.norm(Z, axis=1, keepdims=True)
C = Z @ Z.T
subj = info.subject_id.to_numpy(dtype=str)
same = subj[:, None] == subj[None, :]
off = ~np.eye(len(res), dtype=bool)
print(f"Correlation of pre-pulse images, files of the same subject: min {C[same & off].min():.3f}, "
      f"median {np.median(C[same & off]):.3f} ({(same & off).sum() // 2} pairs)")
print(f"Correlation of pre-pulse images, files of different subjects: max {C[~same].max():.3f}, "
      f"median {np.median(C[~same]):.3f}")
dups = [(info.path[i], info.path[j]) for i in range(len(res)) for j in range(i + 1, len(res))
        if np.array_equal(res[i]["pre_small"], res[j]["pre_small"])]
print("Pairs of files with identical pre-pulse images:", dups if dups else "none")
order = info.sort_values("session_start")
print("\nRecording order within each subject (voltages in order of session start):")
for s, g in order.groupby("subject_id", sort=False):
    print(f"  {s}: " + ", ".join(f"{v:.0f} V" for v in g.volts) + f"   cells per file {g.cells.tolist()}")

A subject is a field of view: the files of one subject show the same image (correlation 0.86 or more)
and different subjects differ (0.43 at most). No file is a copy of another. Each field of view was
recorded first without a pulse and then at escalating voltages, so the same cells were exposed up to
seven times. For the trains this disagrees with Section 3.6 ("Each cell was tested only once"); for
single pulses it agrees with the Results' "escalating field strengths". Cells are treated as
independent across files of one field of view, as each recording has its own F0; results are broken
down by field of view.

In [ ]:
# Cells detected, and what was dropped.
seg = pd.concat([r["cells"].assign(file=i, set=r["set"]) for i, r in enumerate(res)])
print("Objects found by the segmentation:", len(seg))
print("  dropped as smaller than", fluo4.MIN_AREA, "px:", int((seg.area < fluo4.MIN_AREA).sum()))
print("  dropped as touching the border:", int((seg.border & (seg.area >= fluo4.MIN_AREA)).sum()))
print("  kept:", int(seg.kept.sum()))
k = seg[seg.kept]
print("Kept cells, equivalent radius (px): median %.1f, range %.1f to %.1f" % (
    k.radius_px.median(), k.radius_px.min(), k.radius_px.max()))
print("Fraction of each image used as cell-free background: %.2f to %.2f" % (
    min(r["bg_fraction"] for r in res), max(r["bg_fraction"] for r in res)))
print("\nCell exposures by set and voltage (one per cell per file):")
print(info.pivot_table(index="set", columns="volts", values="cells", aggfunc="sum", margins=True).to_string())
first = order.groupby("subject_id").first()
print("\nCells per field of view, counted in its first (sham) recording:")
print(first.groupby("set").cells.agg(["sum", "min", "max", "size"]).to_string())

Of 608 objects, 76 were dropped (69 too small, 7 on the border). The single-pulse session holds 27
distinct cells in 6 fields of view (139 cell recordings); the paper averaged "across 14 cells" (Fig. 6A
legend), which does not match either count, and the dataset does not say which 14. The train session
holds 82 distinct cells in 9 fields of view (311 cell exposures to trains); the paper reports 224 cells,
each tested once.

In [ ]:
cells = pd.concat([pd.DataFrame({
    "file": i, "set": r["set"], "fov": info.fov[i], "volts": r["volts"], "cell": np.arange(len(r["F0"])),
    "F0": r["F0"], "end": fluo4.end_value(r["dF"]), "peak": fluo4.peak_value(r["dF"], r["pulse_frame"])})
    for i, r in enumerate(res)], ignore_index=True)
print("F0 (background-corrected camera counts) by set: median %s, range %s" % (
    cells.groupby("set").F0.median().round(1).to_dict(),
    cells.groupby("set").F0.agg(["min", "max"]).round(1).to_dict("index")))
print("Background (cell-free median, counts), range over files and frames:",
      {s: (round(min(r["bg"].min() for r in res if r["set"] == s), 1),
           round(max(r["bg"].max() for r in res if r["set"] == s), 1)) for s in ["single", "train"]})
print("dF range over all cells and frames (%):", round(min(r["dF"].min() for r in res), 1),
      "to", round(max(r["dF"].max() for r in res), 1))
for s in ["single", "train"]:
    sham = np.concatenate([r["dF"] for r in res if r["set"] == s and r["volts"] == 0])
    pf = next(r["pulse_frame"] for r in res if r["set"] == s)
    print(f"Sham ({s}), {len(sham)} cells: mean dF at frame 0 {sham[:, 0].mean():.1f}%, "
          f"at the pulse frame {sham[:, pf].mean():.1f}%, over the last {fluo4.END_FRAMES} frames "
          f"{sham[:, -fluo4.END_FRAMES:].mean():.1f}%")

Without a pulse, dF falls steadily through the recording, to -12% (single-pulse session) and -16%
(train session) over the last 10 frames, with F0 taken before the pulse as Methods 2.4 say. The next cell
shows where the decline comes from.

In [ ]:
# Where does the decline of unpulsed cells come from? Per set and voltage: the end value (last 10
# frames) relative to the pre-pulse mean of the background-corrected cell signal (dF), of the raw
# cell signal without background subtraction, of the cell-free background, and of the whole frame.
def rel_end(x, pf):
    x = np.asarray(x, float)
    return 100 * (x[..., -fluo4.END_FRAMES:].mean(-1) / x[..., :pf].mean(-1) - 1)

rows = []
for s in ["single", "train"]:
    for v in sorted(info.volts[info.set == s].unique()):
        rs = [r for r in res if r["set"] == s and r["volts"] == v]
        pf = rs[0]["pulse_frame"]
        Fc = [r["F0"][:, None] * (1 + r["dF"] / 100) for r in rs]          # background-corrected F
        raw = [F + r["bg"][None] for F, r in zip(Fc, rs)]                   # F before background subtraction
        rows.append({"set": s, "volts": v, "files": len(rs), "cells": sum(len(F) for F in Fc),
                     "dF_end": np.concatenate([rel_end(F, pf) for F in Fc]).mean(),
                     "raw_cell_end": np.concatenate([rel_end(F, pf) for F in raw]).mean(),
                     "background_end": np.mean([rel_end(r["bg"], pf) for r in rs]),
                     "whole_frame_end": np.mean([rel_end(r["frame_mean"], pf) for r in rs]),
                     "background_over_F0": np.mean([r["bg"][:pf].mean() / r["F0"].mean() for r in rs])})
print("End value relative to the pre-pulse mean (%), by set and voltage:")
print(pd.DataFrame(rows).round(2).to_string(index=False))

# Decline before the pulse (no pulse has yet been given in that recording), by recording order
# within the field of view: slope of each cell's dF over the pre-pulse frames, in % per minute.
rec = order.assign(order_in_fov=order.groupby("subject_id").cumcount() + 1)
pre = []
for i, r in enumerate(res):
    pf, t_min = r["pulse_frame"], np.arange(r["pulse_frame"]) / r["rate_hz"] / 60
    for c in range(len(r["dF"])):
        pre.append({"set": r["set"], "volts": r["volts"], "order_in_fov": int(rec.order_in_fov[i]),
                    "slope_pct_per_min": np.polyfit(t_min, r["dF"][c, :pf], 1)[0]})
pre = pd.DataFrame(pre)
print("\nPre-pulse slope of dF (% per min), by recording order within the field of view (1 = the sham):")
print(pre.groupby(["set", "order_in_fov"]).slope_pct_per_min.agg(["size", "mean", "sem"]).round(2).unstack(0).to_string())

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, s in zip(axes, ["single", "train"]):
    rs = [r for r in res if r["set"] == s and r["volts"] == 0]
    pf, t = rs[0]["pulse_frame"], np.arange(rs[0]["shape"][0]) / rs[0]["rate_hz"]
    Fc = np.concatenate([r["F0"][:, None] * (1 + r["dF"] / 100) for r in rs])
    raw = np.concatenate([r["F0"][:, None] * (1 + r["dF"] / 100) + r["bg"][None] for r in rs])
    bg = np.array([r["bg"] for r in rs])
    for y, lab in [(Fc, "dF, background-corrected"), (raw, "raw cell signal"), (bg, "background")]:
        ax.plot(t, 100 * (y.mean(0) / y[:, :pf].mean() - 1), label=lab)
    ax.axhline(0, color="k", lw=0.5, ls=":")
    ax.set_title(f"sham, {s} session ({len(Fc)} cells)", fontsize=9)
    ax.set_xlabel("time (s)")
axes[0].set_ylabel("change from pre-pulse mean (%)")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()
print("Plotted range (%):", {s: (round(float(d.min()), 1), round(float(d.max()), 1))
                             for s, d in [("single", np.concatenate([r["dF"] for r in res if r["set"] == "single" and r["volts"] == 0]).mean(0)),
                                          ("train", np.concatenate([r["dF"] for r in res if r["set"] == "train" and r["volts"] == 0]).mean(0))]})

The decline is in the cells' own fluorescence, not in the background handling. In the sham recordings
the raw cell signal, before any background is subtracted, falls by 8.7% (single) and 7.5% (train), while
the cell-free background falls by 1.0% and 0.6%. Subtracting the background enlarges the relative decline
(to -12.0% and -15.7%), because the background is 0.45 (single) to 1.26 (train) times the
background-corrected cell signal; no choice of background region removes it, and dropping the subtraction,
which Methods 2.4 prescribe, still leaves -8.7%. Fig. 6A of the paper shows the 0 V/cm mean trace rising to
about +1% by 300 s, and the Fig. 6B traces are flat near zero, so the recordings or the processing behind the
figure differ from these files in a way the paper does not describe (a question for the authors, see the
summary).

The decline is also steepest in the first recording of a field of view and smaller in later ones: before the pulse, dF falls by 3.97% per
minute in the first (sham) recording of the single-pulse session and by 1.46 to 0.41% per minute in the fifth
to seventh; in the train session by 11.4% per minute in the first and 2.0 to 5.4% per minute later, without a steady
trend after the first. As each
field of view was recorded at escalating voltages, voltage and recording order go together, and any end
value compared with the sham, which is always first, carries part of this difference. The step of dF at the
pulse (shown for Fig. 6A below) does not.

In [ ]:
# Segmentation check: the regions on the pre-pulse mean image of each field of view's first recording
# (images shown at the 4-fold reduced size kept while loading).
firsts = order.groupby("subject_id", sort=False).head(1)
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
for ax, (i, row) in zip(axes.ravel(), firsts.iterrows()):
    r = res[i]
    ax.imshow(r["pre_small"], cmap="gray")
    for c in r["cells"].itertuples():
        col = "lime" if c.kept else "red"
        ax.add_patch(plt.Circle((c.x / 4, c.y / 4), c.radius_px / 4, fill=False, color=col, lw=0.6))
        if c.kept:
            ax.add_patch(plt.Circle((c.x / 4, c.y / 4), fluo4.ROI_FRACTION * c.radius_px / 4, fill=False,
                                    color="yellow", lw=0.6))
    ax.set_title(f"{row.subject_id} ({row.set}): {int(r['cells'].kept.sum())} kept", fontsize=8)
    ax.axis("off")
for ax in axes.ravel()[len(firsts):]:
    ax.axis("off")
plt.tight_layout()
plt.show()
print("Green: object outline as a circle of its equivalent radius (kept); red: dropped; yellow: the region used.")

The regions sit on the cells in every field of view; the dropped objects are small specks and
objects cut by the image border.

In [ ]:
# Does the response start at the stated pulse time? Step = mean dF over W frames after a putative
# pulse frame minus the W frames before it, strongly pulsed files minus sham files, with the
# putative pulse frame shifted.
def step(r, at, w):
    m = r["dF"].mean(0)
    return m[at:at + w].mean() - m[at - w:at].mean()

for s, strong, w in [("single", 350, 10), ("train", 230, 5)]:
    rs = [r for r in res if r["set"] == s]
    pf, n = rs[0]["pulse_frame"], rs[0]["shape"][0]
    shifts = np.arange(-pf + w, n - pf - w)
    contrast = [np.mean([step(r, pf + d, w) for r in rs if r["volts"] >= strong])
                - np.mean([step(r, pf + d, w) for r in rs if r["volts"] == 0]) for d in shifts]
    best = shifts[int(np.argmax(contrast))]
    rise = [int(np.argmax(np.diff(r["dF"].mean(0)))) + 1 for r in rs if r["volts"] >= strong]
    print(f"{s}: stated pulse at frame {pf} ({fluo4.PULSE_S[s]:.0f} s). Contrast (>= {strong} V minus sham) "
          f"at the stated frame {contrast[list(shifts).index(0)]:.2f}%, largest at shift {best:+d} frames "
          f"({max(contrast):.2f}%); at shifts -{w}, +{w}: "
          f"{contrast[list(shifts).index(-w)]:.2f}%, {contrast[list(shifts).index(w)]:.2f}%")
    print(f"   frame of steepest rise of the mean trace in the {len(rise)} files at >= {strong} V: "
          f"{sorted(rise)}")

The recordings share the clock the descriptions give: the difference between strongly pulsed and sham
files in the step of dF across a putative pulse frame is largest within one frame of the stated pulse
for single pulses (frame 60 against 61), and three frames after the train onset for trains (the train
lasts 10 s, five frames). With the pulse frame shifted 10 frames earlier (single) or 5 earlier (train),
the contrast is near zero. This is a check of the stated timing against the responses; it is consistent
with the data in both sessions.

## Field strengths

The paper gives field strengths (Sim4Life values 5 um above the glass, Methods 2.2); the files give only
the generator voltage, and neither states the conversion. Two readings are run (author, step 1.6):

- M1, 3.9 V/cm per V, adopted as the main reading (an assumption, see below). The single-pulse voltages
  150, 200, 250, 300 and 350 V then give 585, 780, 975, 1170 and 1365 V/cm, the field series the paper
  uses throughout (570, 780, 1000, 1200, 1400 V/cm), and the train voltages 70, 150, 170, 200, 230 and
  250 V fall nearest the six fields labelled in Fig. 6B (230, 570, 650, 780, 900, 1000 V/cm).
- M2, 3.3 V/cm per V, a variant. The train voltages 70, 200 and 230 V then give 231, 660 and 759 V/cm,
  the 230, 650 and 780 V/cm named for the trains in Section 3.6.

A field the paper names is assigned to the voltage whose field under the reading is nearest. Neither
factor follows from the electrode gap (V/d is 10 V/cm per V for the 1.00 mm the files state and 6.25 V/cm
per V for the 1.6 mm of Methods 2.2), as the paper's fields are simulated at the cells.

In [ ]:
print("Field strength (V/cm) of each voltage under the two readings of the conversion:")
volts = sorted(info.volts.unique())
print(pd.DataFrame({m: fluo4.field(volts, m).round() for m in fluo4.FIELD_PER_VOLT}, index=volts).T.to_string())
avail = {s: sorted(info.volts[info.set == s].unique()) for s in ["single", "train"]}
named = {"single": [780, 1000, 1200], "train": [230, 570, 650, 780, 900, 1000]}
for s in named:
    for m in fluo4.FIELD_PER_VOLT:
        print(f"  {s}, {m}: " + ", ".join(
            f"{f} V/cm -> {fluo4.voltage_for(f, m, avail[s]):.0f} V "
            f"({fluo4.field(fluo4.voltage_for(f, m, avail[s]), m):.0f} V/cm)" for f in named[s]))


def group_table(d, value):
    """Per voltage: n, mean, s.e., one-sample t-test against 0 and Student's t-test against sham."""
    sham = d[d.volts == 0][value]
    rows = []
    for v, g in d.groupby("volts"):
        x = g[value]
        one = stats.ttest_1samp(x, 0)
        two = stats.ttest_ind(x, sham) if v != 0 else None
        rows.append({"volts": v, "n": len(x), "mean": x.mean(), "se": x.sem(),
                     "t_vs_0": one.statistic, "p_vs_0": one.pvalue,
                     "diff_vs_sham": x.mean() - sham.mean() if v != 0 else np.nan,
                     "p_vs_sham": two.pvalue if two else np.nan})
    return pd.DataFrame(rows).set_index("volts")


# Fig. 6B: the fields labelled on its panels and the n printed below each (the paper's figure, a
# reference for the comparison, not a result).
FIG6B_N = {230: 28, 570: 37, 650: 34, 780: 54, 900: 35, 1000: 36}
n_by_volts = info[info.set == "train"].groupby("volts").cells.sum()
print("\nFig. 6B groups against the dataset's train groups under each reading (nearest voltage):")
rows = []
for m in fluo4.FIELD_PER_VOLT:
    for f, n_paper in FIG6B_N.items():
        v = fluo4.voltage_for(f, m, avail["train"])
        rows.append({"mapping": m, "field_fig6b": f, "n_fig6b": n_paper, "volts": v,
                     "field_of_volts": round(fluo4.field(v, m)), "n_dataset": int(n_by_volts[v])})
g6b = pd.DataFrame(rows)
print(g6b.to_string(index=False))
for m, g in g6b.groupby("mapping"):
    print(f"  {m}: distinct voltages used {g.volts.nunique()} of 6; sum of |n_dataset - n_fig6b| = "
          f"{int((g.n_dataset - g.n_fig6b).abs().sum())}; total n {int(g.drop_duplicates('volts').n_dataset.sum())} "
          f"(Fig. 6B {sum(FIG6B_N.values())})")
print("Train voltages in the dataset that no Fig. 6B field takes under M1:",
      [v for v in avail["train"] if v > 0 and v not in set(g6b[g6b.mapping == "M1"].volts)])

Under M1 the six Fig. 6B fields take six different train voltages, and the dataset's group sizes are
close to the figure's n (equal at 230 V/cm, and within 3 at 570, 650 and 1000 V/cm; larger at 780 and 900
V/cm). Under M2, 900 and 1000 V/cm fall on 250 and 300 V, and the 300 V group has 6 cells where the figure
has 36; the six M2 groups happen to sum to 224, the paper's total, while the M1 groups sum to 256. The
same electrodes and generator served both sessions (Methods 2.2), so M1 is adopted for both
sessions, as an assumption; M2 is still shown for every claim. The train voltages 100, 180, 190, 210, 220
and 300 V are in no Fig. 6B group under M1.

## fig6a-ca-entry-1000-1200

> A single 1-us pulse causes clear Ca2+ entry into CHO cells at 1000 and 1200 V/cm, among field
> strengths escalated from 0 to 1200 V/cm.

Criterion (claims file): per cell dF as in Methods 2.4; reproduced when, under at least one reading, the
post-pulse dF at both 1000 and 1200 V/cm is significantly above zero (one-sample t-test) or above the
0 V/cm group (two-tailed Student's t-test) at p < 0.05, and the group means at 1000 and 1200 V/cm exceed
those of every field of 650 V/cm or less. The post-pulse value is read as (a) the mean of the last 10
frames or (b) each cell's peak after the pulse. Both conversions are run.

In [ ]:
A = cells[cells.set == "single"]
tabs_A = {v: group_table(A, v) for v in ["end", "peak"]}
for v, t in tabs_A.items():
    print(f"Single pulses, per-cell {v} dF (%), by voltage:")
    print(t.round(4).to_string(), "\n")
rows = []
for m in fluo4.FIELD_PER_VOLT:
    v1000, v1200 = (fluo4.voltage_for(f, m, avail["single"]) for f in (1000, 1200))
    low = [v for v in avail["single"] if fluo4.field(v, m) <= 650]
    for value, t in tabs_A.items():
        exceed = all(t.loc[v, "mean"] > t.loc[low, "mean"].max() for v in (v1000, v1200))
        for test in ["one-sample vs 0", "Student vs 0 V/cm"]:
            if test.startswith("one"):
                sig = [t.loc[v, "p_vs_0"] < 0.05 and t.loc[v, "mean"] > 0 for v in (v1000, v1200)]
                ps = [t.loc[v, "p_vs_0"] for v in (v1000, v1200)]
            else:
                sig = [t.loc[v, "p_vs_sham"] < 0.05 and t.loc[v, "diff_vs_sham"] > 0 for v in (v1000, v1200)]
                ps = [t.loc[v, "p_vs_sham"] for v in (v1000, v1200)]
            rows.append({"mapping": m, "value": value, "test": test, "V_1000": v1000, "V_1200": v1200,
                         "low_fields_V": low, "mean_1000": t.loc[v1000, "mean"], "mean_1200": t.loc[v1200, "mean"],
                         "max_low_mean": t.loc[low, "mean"].max(), "p_1000": ps[0], "p_1200": ps[1],
                         "sig_1000": sig[0], "sig_1200": sig[1], "exceed_low": exceed,
                         "holds_both": sig[0] and sig[1] and exceed,
                         "holds_one": (sig[0] or sig[1]) and exceed})
readings_6a = pd.DataFrame(rows)
print(readings_6a.round(6).to_string())
print("\np-values of the table above, in scientific notation:")
print(readings_6a[["mapping", "value", "test", "p_1000", "p_1200"]].to_string(float_format="%.2e"))
for m in fluo4.FIELD_PER_VOLT:
    r = readings_6a[readings_6a.mapping == m]
    print(f"{m}: readings holding at both fields: {int(r.holds_both.sum())} of {len(r)}; at one or both: "
          f"{int(r.holds_one.sum())}")

In [ ]:
print("Per field of view: mean end and peak dF (%) by voltage, single pulses")
print(A.pivot_table(index="fov", columns="volts", values="end", aggfunc="mean").round(2).to_string())
print(A.pivot_table(index="fov", columns="volts", values="peak", aggfunc="mean").round(2).to_string())
print(A.pivot_table(index="fov", columns="volts", values="end", aggfunc="size").to_string())

fig, axes = plt.subplots(1, len(avail["single"]), figsize=(18, 3.2), sharey=True)
lims = (-25, 40)
t_a = np.arange(300)
outside = 0
for ax, v in zip(axes, avail["single"]):
    tr = np.concatenate([r["dF"] for r in res if r["set"] == "single" and r["volts"] == v])
    outside += int(((tr < lims[0]) | (tr > lims[1])).any(1).sum())
    ax.plot(t_a, tr.T, color="tab:blue", lw=0.4, alpha=0.5)
    mu, se = tr.mean(0), tr.std(0, ddof=1) / np.sqrt(len(tr))
    ax.fill_between(t_a, mu - se, mu + se, color="0.6")
    ax.plot(t_a, mu, color="tab:orange", lw=2)
    ax.axvline(60, color="k", lw=0.8)
    ax.axhline(0, color="k", lw=0.5, ls=":")
    ax.set_title(f"{v:.0f} V\nM1 {fluo4.field(v, 'M1'):.0f}, M2 {fluo4.field(v, 'M2'):.0f} V/cm", fontsize=9)
    ax.set_xlabel("time (s)")
axes[0].set_ylim(*lims)
axes[0].set_ylabel("Fluo-4 dF (%)")
plt.tight_layout()
plt.show()
print(f"Axis {lims}; cell traces with points outside it: {outside}")

In [ ]:
# Variants (not the criterion's readings), against two confounds of the end and peak values: the
# decline of the signal, and the recording order (the sham is always recorded first, and the decline
# is steepest in the first recording, see the data check).
# (1) Step at the pulse: each cell's mean dF over the 5 frames after the pulse less the 5 before it.
A = A.assign(step=[s for r in res if r["set"] == "single" for s in
                   (r["dF"][:, r["pulse_frame"]:r["pulse_frame"] + 5].mean(1)
                    - r["dF"][:, r["pulse_frame"] - 5:r["pulse_frame"]].mean(1))])
print("Step at the pulse (%), single pulses, by voltage:")
print(group_table(A, "step").round(4).to_string())
print("p-values:", group_table(A, "step")[["p_vs_0", "p_vs_sham"]].to_string(float_format="%.2e"))
# (2) Fields of view as the unit: per field of view, mean at the voltage less mean at its own sham,
# one-sample t-test over fields of view.
rows = []
for value in ["end", "peak", "step"]:
    fm = A.pivot_table(index="fov", columns="volts", values=value, aggfunc="mean")
    for v in avail["single"][1:]:
        d = (fm[v] - fm[0]).dropna()
        rows.append({"value": value, "volts": v, "n_fov": len(d), "mean_diff_vs_own_sham": d.mean(),
                     "min_diff": d.min(), "p_one_sample": stats.ttest_1samp(d, 0).pvalue if len(d) > 1 else np.nan})
print("\nPer field of view, value at the voltage less its own sham (t-test over fields of view):")
print(pd.DataFrame(rows).round(4).to_string(index=False))
# (3) Fig. 6A averages 14 cells: cells per field of view and voltage, and the voltage groups' sizes.
n_fv = A.pivot_table(index="fov", columns="volts", values="end", aggfunc="size")
print("\nCells per voltage group:", A.groupby("volts").size().to_dict(),
      "; cells per field of view:", n_fv[0].astype(int).to_dict())
from itertools import combinations
sizes = n_fv[0].astype(int)
print("Sets of fields of view whose cells sum to 14:",
      [c for k in range(1, len(sizes) + 1) for c in combinations(sizes.index, k) if sizes[list(c)].sum() == 14])

**Result: reproduced, under both conversions.** Under M1 (250 and 300 V), three of the four readings
hold at both fields: the end value against the 0 V/cm group (p = 4.9e-09 and 3.0e-11), and the peak against
zero or against the 0 V/cm group (means 2.30 and 5.35% against at most 0.55% at 650 V/cm or less). The
end value against zero fails: it is below zero at both fields (-5.44 and -3.83%), as is the sham (-12.0%).
Under M2 (300 and 350 V) the same three readings hold. In every field of view the peak at 250 V is above
the peak at 0 V, and at 300 V above the peaks at 0 and 150 V. With fields of view as the unit, the peak
less the same field of view's sham is 2.42% at 250 V (4 fields of view, p = 0.0004) and 5.02% at 300 V (5,
p = 0.0002).

The end and peak readings compare later recordings with the sham, recorded first, while the decline is steepest in the first recording (data check). The step at the pulse, which that does not affect, is above the sham's at
250 V (0.35 +- 0.17%, against the sham p = 0.025) and 300 V (1.78 +- 0.31%, p = 3.2e-07), and not at 150
or 200 V (p = 0.13 and 0.10); at 250 V it is small and not above zero on its own (p = 0.058). The entry at
1000 V/cm under M1 is therefore a small one, a few tenths of a percent at the pulse, and clearer at 1200
V/cm (1.78%) and above (3.94% at 350 V). The single-pulse session holds 27 cells in 6 fields of view, and
three sets of fields of view give 14 cells, so which 14 cells Fig. 6A averages cannot be told from the data.

## fig6a-ca-sustained

> After a single 1-us pulse at 1000 or 1200 V/cm the Ca2+ signal stays elevated through the 5-min
> observation period.

Criterion (claims file): at both 1000 and 1200 V/cm, the mean dF over the last 10 frames is above zero
by a one-sample t-test at p < 0.05; partially reproduced when this holds at one of the two. The ratio of
the final to the peak group mean is reported.

In [ ]:
rows = []
for m in fluo4.FIELD_PER_VOLT:
    for f in (1000, 1200):
        v = fluo4.voltage_for(f, m, avail["single"])
        t = tabs_A["end"]
        tr = np.concatenate([r["dF"] for r in res if r["set"] == "single" and r["volts"] == v]).mean(0)
        one_sided = stats.ttest_1samp(A[A.volts == v].end, 0, alternative="greater").pvalue
        rows.append({"mapping": m, "field": f, "volts": v, "n": t.loc[v, "n"], "end_mean": t.loc[v, "mean"],
                     "end_se": t.loc[v, "se"], "p_vs_0_two_sided": t.loc[v, "p_vs_0"],
                     "p_above_0_one_sided": one_sided, "above_zero": t.loc[v, "mean"] > 0 and t.loc[v, "p_vs_0"] < 0.05,
                     "peak_of_mean_trace": tr[60:].max(), "final_over_peak": tr[-fluo4.END_FRAMES:].mean() / tr[60:].max(),
                     "end_minus_sham": t.loc[v, "diff_vs_sham"], "p_vs_sham": t.loc[v, "p_vs_sham"]})
print(pd.DataFrame(rows).round(6).to_string())
print("p-values:")
print(pd.DataFrame(rows)[["mapping", "field", "volts", "p_vs_0_two_sided", "p_above_0_one_sided", "p_vs_sham"]]
      .to_string(float_format="%.2e"))

In [ ]:
# Variant (not the claim's criterion): traces less the mean sham trace of the same field of view.
sham_by_fov = {info.fov[i]: r["dF"].mean(0) for i, r in enumerate(res) if r["set"] == "single" and r["volts"] == 0}
rows = []
for v in avail["single"][1:]:
    corr = np.concatenate([r["dF"] - sham_by_fov[info.fov[i]] for i, r in enumerate(res)
                           if r["set"] == "single" and r["volts"] == v])
    e = corr[:, -fluo4.END_FRAMES:].mean(1)
    rows.append({"volts": v, "n": len(e), "end_minus_own_sham": e.mean(), "se": e.std(ddof=1) / np.sqrt(len(e)),
                 "p_vs_0": stats.ttest_1samp(e, 0).pvalue})
print("End dF less the same field of view's sham trace (variant):")
print(pd.DataFrame(rows).set_index("volts").to_string(float_format="%.3g"))

**Result: not reproduced, under both conversions.** The end-of-recording dF is below zero at 1000 and
1200 V/cm under M1 (-5.44 and -3.83%) and at 1000 V/cm under M2 (-3.83%); at 1200 V/cm under M2 (350 V)
it is +1.24% with p = 0.53. The final over peak ratio of the group mean trace is -63 and -2.4 under M1.
The decline is the one the sham shows, and it is in the cells' raw fluorescence (-8.7% in the sham), not in
the background (-1.0%), so it does not come from how the background was taken (data check). Against the
0 V/cm group the end value is higher by 6.6 and 8.2% (p = 4.9e-09 and 3.0e-11), and less the same field of view's sham
trace it is above zero at every voltage of 200 V or more (4.8 to 12.9%, p below 0.001), but these
comparisons include the change of the decline with recording order. So the signal stays above that of
unpulsed cells to the end, but not above its own pre-pulse level, which is what the criterion tests. The
paper's Fig. 6A shows no such decline in its 0 V/cm trace, so the outcome rests on a difference between these
recordings and the figure that the paper does not explain.

## fig6a-780-not-significant

> After a single 1-us pulse at 780 V/cm the Ca2+ signal rises only subtly, at the detection limit, and
> not significantly.

Criterion (claims file): a claim of absence, judged on a one-sample t-test of each cell's end-of-recording
dF (last 10 frames) against zero at p < 0.05: reproduced when it does not find an increase. The t-test
against the 0 V/cm group and each cell's peak are variants.

In [ ]:
rows = []
for m in fluo4.FIELD_PER_VOLT:
    v = fluo4.voltage_for(780, m, avail["single"])
    for value in ["end", "peak", "step"]:
        x, sham = A[A.volts == v][value], A[A.volts == 0][value]
        rows.append({"mapping": m, "volts": v, "field": fluo4.field(v, m), "value": value, "n": len(x),
                     "mean": x.mean(), "se": x.sem(),
                     "p_increase_one_sample": stats.ttest_1samp(x, 0, alternative="greater").pvalue,
                     "p_two_sided_one_sample": stats.ttest_1samp(x, 0).pvalue,
                     "diff_vs_sham": x.mean() - sham.mean(), "p_vs_sham_two_tailed": stats.ttest_ind(x, sham).pvalue})
t780 = pd.DataFrame(rows)
print(t780.round(6).to_string())
main = t780[(t780.value == "end")]
print("\nMain reading (one-sample t-test of the end value against zero, p < 0.05 for an increase):")
for _, r in main.iterrows():
    print(f"  {r.mapping}: {r.volts:.0f} V, mean {r['mean']:.2f} +- {r.se:.2f}%, one-sided p = {r.p_increase_one_sample:.3g}"
          f" -> increase found: {r['mean'] > 0 and r.p_two_sided_one_sample < 0.05}")
x200, x250 = A[A.volts == 200].step, A[A.volts == 250].step
print(f"\nStep at 250 V against the step at 200 V (Student's t-test, two-tailed): p = {stats.ttest_ind(x250, x200).pvalue:.3g}")
print("\nVariant reading that detects 1000 and 1200 V/cm under M1: step at the pulse against the 0 V/cm group "
      "(Student's t-test, two-tailed):")
for v in (200, 250, 300):
    x, sham = A[A.volts == v].step, A[A.volts == 0].step
    print(f"  {v} V (M1 {fluo4.field(v, 'M1'):.0f} V/cm): {x.mean():.2f} +- {x.sem():.2f}% against {sham.mean():.2f}%, "
          f"p = {stats.ttest_ind(x, sham).pvalue:.3g}")

**Result: reproduced, under both conversions.** The one-sample test finds no increase at 780 V/cm: the
end value is -7.42 +- 0.52% under M1 (200 V, n = 24) and -5.44 +- 0.55% under M2 (250 V, n = 18), one-sided
p = 1 for an increase in both. This test cannot detect an increase here: it also finds none at 1000 and
1200 V/cm under M1, where the end values are -5.44 and -3.83%. Of the variants, the end value against the
0 V/cm group (higher by 4.60%, p = 1.3e-06) and the peak (1.76 +- 0.61%, against zero p = 0.0086, against
the sham peak p = 0.021) find an effect at 780 V/cm under M1, and both also carry the order effect of the
data check. The step at the pulse, which separates 1000 and 1200 V/cm from the sham (p = 0.025 and 3.2e-07), finds none at 780 V/cm under M1 (0.15 +- 0.16%, p = 0.10); under M2, where 780 V/cm is 250 V, it finds
one (0.35 +- 0.17%, p = 0.025). The step is the only variant not confounded by recording order, but it is weak evidence here: it
separates 1000 V/cm from the sham only barely (p = 0.025, and p = 0.058 against zero), and the steps at
780 and 1000 V/cm under M1 (0.15 +- 0.16 and 0.35 +- 0.17%) do not differ (p = 0.40). The end and
peak readings against the sham, which also separate 1000 and 1200 V/cm from it, find an effect at 780 V/cm.
So the outcome follows the claims file's test, which says little here. A
question on the criterion's test is asked of a reader (questions/001521.yaml).

## fig6b-train-threshold

> With trains of sixty 1-us pulses at 6 Hz (each cell tested once, 224 cells), a strong Ca2+ response
> occurs in 28% of cells at 780 V/cm and in nearly all cells at higher fields.

Criterion (claims file): a strong response is (a) an end-of-recording dF above the largest among the
0 V/cm (or lowest field) cells, or (b) above 3 s.d. of that group, read here as above its mean plus 3
s.d.; reproduced when, under one reading, the fraction at 780 V/cm is within 28 +- 2.8% and at least 90%
respond at every field above 780 V/cm. Expected: 224 cells tested with trains.

In [ ]:
B = cells[cells.set == "train"]
sham_B, low_B = B[B.volts == 0].end, B[B.volts == min(v for v in avail["train"] if v > 0)].end
rules = {"a: > max of 0 V/cm": sham_B.max(), "a: > max of lowest field": low_B.max(),
         "b: > mean + 3 s.d. of 0 V/cm": sham_B.mean() + 3 * sham_B.std(),
         "b: > mean + 3 s.d. of lowest field": low_B.mean() + 3 * low_B.std(),
         "b literal: > 3 s.d. of 0 V/cm": 3 * sham_B.std(),
         "b literal: > 3 s.d. of lowest field": 3 * low_B.std()}
print("Thresholds on the end-of-recording dF (%):", {k: round(v, 2) for k, v in rules.items()})
frac = pd.DataFrame({k: B.groupby("volts").end.apply(lambda x, t=t: 100 * (x > t).mean()) for k, t in rules.items()})
frac.insert(0, "n", B.groupby("volts").size())
for m in fluo4.FIELD_PER_VOLT:
    frac[f"field_{m}"] = fluo4.field(frac.index, m).round()
print(frac.round(1).to_string())
rows = []
for m in fluo4.FIELD_PER_VOLT:
    v780 = fluo4.voltage_for(780, m, avail["train"])
    # the fields above 780 V/cm in Fig. 6B are 900 and 1000 V/cm
    higher = sorted({fluo4.voltage_for(f, m, avail["train"]) for f in (900, 1000)})
    for k in rules:
        rows.append({"mapping": m, "rule": k, "V_780": v780, "pct_780": frac.loc[v780, k],
                     "in_tolerance": abs(frac.loc[v780, k] - 28) <= 2.8, "higher_V": higher,
                     "min_pct_higher": frac.loc[higher, k].min(), "all_higher_ge_90": (frac.loc[higher, k] >= 90).all()})
readings_6b = pd.DataFrame(rows)
print(readings_6b.round(1).to_string())
n_exp = int((B.volts > 0).sum())
print(f"\nCell exposures to trains (non-sham files): {n_exp}; with sham files: {len(B)}; "
      f"distinct cells (cells in each field of view's sham recording): {int(first[first.set == 'train'].cells.sum())}")
print("Cells per voltage group (paper: 28 to 54 per field strength):", B[B.volts > 0].groupby("volts").size().to_dict())
for m in fluo4.FIELD_PER_VOLT:
    vs = sorted(set(g6b[g6b.mapping == m].volts))
    print(f"{m}: cells in the voltage groups of the six Fig. 6B fields {vs}: {int(B.volts.isin(vs).sum())} (paper: 224)")

In [ ]:
# Variant: each field of view's first exposure only, as if each cell had been tested once.
firstexp = (order[(order.set == "train") & (order.volts > 0)].groupby("subject_id").head(1))
fx = B[B.file.isin(firstexp.index)]
print("First train exposure of each field of view:",
      firstexp[["subject_id", "volts", "cells"]].to_string(index=False))
print(pd.DataFrame({k: fx.groupby("volts").end.apply(lambda x, t=t: 100 * (x > t).mean()) for k, t in rules.items()})
      .assign(n=fx.groupby("volts").size()).round(1).to_string())

fig, axes = plt.subplots(1, len(avail["train"]), figsize=(22, 3.2), sharey=True)
lims = (-40, 200)
t_b = np.arange(90) * 2.0
outside = 0
for ax, v in zip(axes, avail["train"]):
    tr = np.concatenate([r["dF"] for r in res if r["set"] == "train" and r["volts"] == v])
    outside += int(((tr < lims[0]) | (tr > lims[1])).any(1).sum())
    ax.plot(t_b, tr.T, color="tab:blue", lw=0.4, alpha=0.5)
    mu, se = tr.mean(0), tr.std(0, ddof=1) / np.sqrt(len(tr))
    ax.fill_between(t_b, mu - se, mu + se, color="0.6")
    ax.plot(t_b, mu, color="tab:orange", lw=2)
    ax.axvspan(30, 40, color="k", alpha=0.15)
    ax.set_title(f"{v:.0f} V, n={len(tr)}\nM1 {fluo4.field(v, 'M1'):.0f}, M2 {fluo4.field(v, 'M2'):.0f}", fontsize=8)
    ax.set_xlabel("time (s)")
axes[0].set_ylim(*lims)
axes[0].set_ylabel("Fluo-4 dF (%)")
plt.tight_layout()
plt.show()
print(f"Axis {lims}; cell traces with points outside it: {outside}")

**Result: not reproduced.** The fields above 780 V/cm are taken as those of Fig. 6B, 900 and 1000 V/cm
(230 and 250 V under M1, 250 and 300 V under M2). With the criterion's end window of 10 frames, under no reading is the fraction at 780 V/cm within 25.2
to 30.8%: it is 2.9 to 22.1% under M1 (200 V, 68 cells) and 44.7 to 78.7% under M2 (230 V, 47 cells), over
the six rules, including "above 3 s.d." read literally (a threshold of 3 s.d. of the reference group). Nor
do 90% respond at both higher fields: the lower of the two fractions is 44.7 to 78.7% under M1 and 61.5 to
79.5% under M2. The six voltage groups that correspond to the Fig. 6B fields hold 256 cells under M1 and
224 under M2, against the paper's 224; in all, the dataset holds 311 cell exposures to trains, of 82 distinct
cells, not 224 cells each tested once. Taking only the first train of each field of view, 9.1 to 18.2%
respond at 200 V and 90.9 to 100% at 250 V (11 cells each). The region-size check below gives the same
outcome in all 36 combinations, and so does a background less affected by dim cells. With an end window of 5
frames and a region of 0.5 of the radius, the 780 V/cm fraction under M1 and the rule "above the largest of the
lowest field" is 27.9%, within tolerance, but the fields above it then reach 76.6% (230 V) and 82.1% (250 V),
not 90%.

## fig6b-650-weak-but-significant

> With 60-pulse trains at 650 V/cm only 2 cells (5.9%) respond strongly, but the mean Ca2+ signal at the
> end of recording is significantly above zero and marginally above that at 230 V/cm.

Expected: 2 strongly responding cells, 5.9 +- 0.59%. The claim has no criterion of its own; a strong
response is as in fig6b-train-threshold. The statement's tests (one-sample t-test against zero, p <
0.001; single-tailed t-test against 230 V/cm, p < 0.05) are run, with the direction of each effect.

In [ ]:
print("Trains, per-cell end dF (%) by voltage: n, mean, s.e., one-sample t-test against 0, and against 70 V "
      "(two-tailed and single-tailed for higher):")
lowest = B[B.volts == 70].end
tb = group_table(B, "end").drop(columns=["diff_vs_sham", "p_vs_sham"])
tb["p_vs_70V_two_tailed"] = [stats.ttest_ind(B[B.volts == v].end, lowest).pvalue if v != 70 else np.nan for v in tb.index]
tb["p_above_70V_one_tailed"] = [stats.ttest_ind(B[B.volts == v].end, lowest, alternative="greater").pvalue
                                if v != 70 else np.nan for v in tb.index]
tb["M1_field"], tb["M2_field"] = fluo4.field(tb.index, "M1").round(), fluo4.field(tb.index, "M2").round()
print(tb.round(5).to_string(), "\n")
rows = []
for m in fluo4.FIELD_PER_VOLT:
    v650, v230 = (fluo4.voltage_for(f, m, avail["train"]) for f in (650, 230))
    x, y = B[B.volts == v650].end, B[B.volts == v230].end
    for k, t in rules.items():
        rows.append({"mapping": m, "V_650": v650, "V_230": v230, "rule": k, "n": len(x),
                     "strong": int((x > t).sum()), "pct_strong": 100 * (x > t).mean()})
    one = stats.ttest_1samp(x, 0)
    print(f"{m}: {v650:.0f} V ({fluo4.field(v650, m):.0f} V/cm), end dF {x.mean():.2f} +- {x.sem():.2f}% (n={len(x)}), "
          f"{'above' if x.mean() > 0 else 'below'} zero; one-sample t-test: t = {one.statistic:.2f}, two-sided p = {one.pvalue:.3g}, "
          f"one-sided p for above zero = {stats.ttest_1samp(x, 0, alternative='greater').pvalue:.3g}; "
          f"against {v230:.0f} V ({fluo4.field(v230, m):.0f} V/cm, {y.mean():.2f}%, n={len(y)}), single-tailed "
          f"Student's t-test for higher p = {stats.ttest_ind(x, y, alternative='greater').pvalue:.3g}")
print(pd.DataFrame(rows).round(1).to_string())

**Result: not reproduced.** The statement's two tests fail under the adopted conversion and one fails
under the other. Under M1 the mean end value at 170 V (650 V/cm) is -9.17 +- 0.83%, significantly below
zero, not above it (one-sided p = 1 for above zero), and not above the 70 V group (single-tailed p = 0.20).
Under M2 the 200 V mean is -1.98 +- 1.77%, not different from zero (p = 0.27), though above the 70 V group
(single-tailed p = 0.0023). The expected count is met under one combination only: M1 (170 V, 36 cells), the
rule "above the largest of the lowest field", and the region of 0.7 of the cell radius (2 cells, 5.6%);
the other five rules give 0 cells under M1, regions of 0.5 and 1.0 of the radius give 1 cell (2.8%) under
the same rule, and M2 (200 V, 68 cells) gives 2 to 15 cells (2.9 to 22.1%). As the tests fail under both
conversions, the outcome does not depend on which conversion or rule is taken. The end values carry the
decline of the data check.

## Region size and end window

The region over each cell is 0.7 of the cell's equivalent radius (a standard choice), and the end value
is the mean of the last 10 frames (the claims' criterion). The key numbers under regions of 0.5 and 1.0 of
the radius and end windows of 5 and 20 frames, then every reading for the trains.

In [ ]:
# Region size and end window are standard choices (claims file, opening comment): the key numbers under others.
def cells_for(roi, frames):
    out = []
    for i, r in enumerate(res):
        d = r["dF"] if roi == fluo4.ROI_FRACTION else r["dF_roi"][roi]
        out.append(pd.DataFrame({"set": r["set"], "volts": r["volts"], "end": fluo4.end_value(d, frames),
                                 "peak": fluo4.peak_value(d, r["pulse_frame"])}))
    return pd.concat(out, ignore_index=True)

rows = []
for roi in (0.5, fluo4.ROI_FRACTION, 1.0):
    for frames in (5, fluo4.END_FRAMES, 20):
        c = cells_for(roi, frames)
        a, b = c[c.set == "single"], c[c.set == "train"]
        sham_a = a[a.volts == 0].end
        sb, lb = b[b.volts == 0].end, b[b.volts == 70].end
        row = {"roi_fraction": roi, "end_frames": frames}
        for v in (0, 150, 200, 250, 300, 350):
            row[f"A_end_{v}V"] = a[a.volts == v].end.mean()
        for v in (250, 300):
            row[f"A_p_vs_sham_{v}V"] = stats.ttest_ind(a[a.volts == v].end, sham_a).pvalue
        for v in (300, 350):
            row[f"A_p_vs_0_{v}V"] = stats.ttest_1samp(a[a.volts == v].end, 0).pvalue
        for v in (170, 200, 230, 250, 300):
            x = b[b.volts == v].end
            row[f"B_pct_{v}V_a0"] = 100 * (x > sb.max()).mean()
            row[f"B_pct_{v}V_alow"] = 100 * (x > lb.max()).mean()
        row["B_n_strong_170V_alow"] = int((b[b.volts == 170].end > lb.max()).sum())
        rows.append(row)
robust = pd.DataFrame(rows)
print(robust.round(3).T.to_string())

For single pulses the group end values move by less than 1% across these choices, and every comparison
above keeps its direction and significance. For the trains, the fractions at 200 V under the rule "above
the largest of the lowest field" range from 11.8 to 30.9%, and the count at 170 V from 0 to 3. One of these, the 5-frame window with a region of 0.5 of the radius, gives
27.9% at 200 V, within 28 +- 2.8%, but 76.6% at 230 V and 82.1% at 250 V, short of the 90% the criterion needs
at the higher fields.

In [ ]:
# Trains: every reading of the conversion, the response rule and the region size, with the end
# window of the criterion (last 10 frames).
rows = []
for roi in (0.5, fluo4.ROI_FRACTION, 1.0):
    c = cells_for(roi, fluo4.END_FRAMES)
    b = c[c.set == "train"]
    sb, lb = b[b.volts == 0].end, b[b.volts == 70].end
    thr = {"a: > max of 0 V/cm": sb.max(), "a: > max of lowest field": lb.max(),
           "b: > mean + 3 s.d. of 0 V/cm": sb.mean() + 3 * sb.std(),
           "b: > mean + 3 s.d. of lowest field": lb.mean() + 3 * lb.std(),
           "b literal: > 3 s.d. of 0 V/cm": 3 * sb.std(), "b literal: > 3 s.d. of lowest field": 3 * lb.std()}
    for m in fluo4.FIELD_PER_VOLT:
        v780, v650 = (fluo4.voltage_for(f, m, avail["train"]) for f in (780, 650))
        higher = sorted({fluo4.voltage_for(f, m, avail["train"]) for f in (900, 1000)})
        for k, t in thr.items():
            x650 = b[b.volts == v650].end
            rows.append({"roi": roi, "mapping": m, "rule": k,
                         "pct_780": 100 * (b[b.volts == v780].end > t).mean(),
                         "min_pct_higher": min(100 * (b[b.volts == v].end > t).mean() for v in higher),
                         "n_650": len(x650), "strong_650": int((x650 > t).sum()),
                         "pct_650": 100 * (x650 > t).mean()})
grid = pd.DataFrame(rows)
grid["fig6b_threshold_holds"] = (abs(grid.pct_780 - 28) <= 2.8) & (grid.min_pct_higher >= 90)
grid["fig6b_650_matches"] = (grid.strong_650 == 2) & (abs(grid.pct_650 - 5.9) <= 0.59)
print(grid.round(1).to_string())
print(f"\nfig6b-train-threshold criterion met in {int(grid.fig6b_threshold_holds.sum())} of {len(grid)} combinations; "
      f"780 V/cm fraction within 28 +- 2.8% in {int((abs(grid.pct_780 - 28) <= 2.8).sum())}")
print(f"fig6b-650 expected values (2 cells, 5.9 +- 0.59%) met in {int(grid.fig6b_650_matches.sum())} of {len(grid)}: "
      + "; ".join(f"{r.mapping}, {r.rule}, region {r.roi}" for r in grid[grid.fig6b_650_matches].itertuples()))

The fig6b-train-threshold criterion is met in none of the 36 combinations of region size, conversion
and rule. The fig6b-650 expected values are met in one.

A variant background: the 10th percentile of the cell-free pixels in place of their median, which lowers the effect of dim cells the threshold misses inside the cell-free region (seen in several train fields of view). The sham decline and the train readings under it:

In [ ]:
# Variant background (fluo4.BG_LOW_PERCENTILE percentile of the cell-free pixels), main region size.
cl = pd.concat([pd.DataFrame({"set": r["set"], "volts": r["volts"], "end": fluo4.end_value(r["dF_bg_low"]),
                              "end_median_bg": fluo4.end_value(r["dF"])}) for r in res], ignore_index=True)
print("Background over F0 (pre-pulse means), median against variant:",
      {s: (round(float(np.mean([r["bg"][:r["pulse_frame"]].mean() / r["F0"].mean() for r in res if r["set"] == s])), 2),
           round(float(np.mean([r["bg_low"][:r["pulse_frame"]].mean() /
                          (r["F0"] + r["bg"][:r["pulse_frame"]].mean() - r["bg_low"][:r["pulse_frame"]].mean()).mean()
                          for r in res if r["set"] == s])), 2)) for s in ["single", "train"]})
print("Mean end dF (%) by set and voltage, median background against variant:")
print(cl.groupby(["set", "volts"])[["end_median_bg", "end"]].mean().unstack(0).to_string(float_format="%.2f"))
b = cl[cl.set == "train"]
sb, lb = b[b.volts == 0].end, b[b.volts == 70].end
thr = {"a: > max of 0 V/cm": sb.max(), "a: > max of lowest field": lb.max(),
       "b: > mean + 3 s.d. of 0 V/cm": sb.mean() + 3 * sb.std(),
       "b: > mean + 3 s.d. of lowest field": lb.mean() + 3 * lb.std(),
       "b literal: > 3 s.d. of 0 V/cm": 3 * sb.std(), "b literal: > 3 s.d. of lowest field": 3 * lb.std()}
rows = []
for m in fluo4.FIELD_PER_VOLT:
    v780, v650 = (fluo4.voltage_for(f, m, avail["train"]) for f in (780, 650))
    higher = sorted({fluo4.voltage_for(f, m, avail["train"]) for f in (900, 1000)})
    for k, t in thr.items():
        x650 = b[b.volts == v650].end
        rows.append({"mapping": m, "rule": k, "pct_780": 100 * (b[b.volts == v780].end > t).mean(),
                     "min_pct_higher": min(100 * (b[b.volts == v].end > t).mean() for v in higher),
                     "strong_650": int((x650 > t).sum()), "pct_650": 100 * (x650 > t).mean(),
                     "mean_650": x650.mean(), "p_650_above_0": stats.ttest_1samp(x650, 0, alternative="greater").pvalue})
gbg = pd.DataFrame(rows)
gbg["fig6b_threshold_holds"] = (abs(gbg.pct_780 - 28) <= 2.8) & (gbg.min_pct_higher >= 90)
gbg["fig6b_650_count_matches"] = (gbg.strong_650 == 2) & (abs(gbg.pct_650 - 5.9) <= 0.59)
print(gbg.round(3).to_string())
print(f"\nfig6b-train-threshold criterion met in {int(gbg.fig6b_threshold_holds.sum())} of {len(gbg)}; "
      f"fig6b-650 count met in {int(gbg.fig6b_650_count_matches.sum())} of {len(gbg)}")

With the variant background, the background falls from 1.81 to 1.21 times the corrected cell signal in the
train session (pre-pulse means over all train files), but the sham still declines, to -13.5% (train) and
-11.3% (single), and the outcomes do not change: the fig6b-train-threshold criterion is met in none of the 12
combinations of conversion and rule (780 V/cm fraction 2.9 to 23.5% under M1 and 42.6 to 91.5% under M2,
with the lower of the higher-field fractions at most 87.2%), the 650 V/cm mean stays below zero (-7.37% under
M1, -1.54% under M2, one-sided p = 1.0 and 0.87 for above zero), and the count of 2 cells (5.9%) is met in none.
So the dim cells inside the cell-free region do not decide these outcomes.

## fig5b-conductivity-calibration

> By Eq. 2, a 10-fold increase of effective membrane conductivity from 0.0003 mS/m reduces the pole IMP
> by only about 2%, and a 100-fold increase by 12%.

Criterion (claims file): a model computation with the paper's values (cytoplasm 0.4 S/m, medium 1.64 S/m,
intact membrane 3e-7 S/m, R 8 um, d 5 nm). The reduction is 1 - f/1.5 (% of IMPC, Fig. 5B legend); f
relative to f at the intact conductivity is a variant. Expected: 2 +- 0.2% and 12 +- 1.2%. This uses no
recordings.

In [ ]:
def f_eq2(lam_m, R=8e-6, d=5e-9, lam_i=0.4, lam_o=1.64):
    """Eq. 2 of the paper, with its constants (Methods 2.3)."""
    num = 3 * lam_o * (3 * R**2 * d * lam_i + (3 * d**2 * R - d**3) * (lam_m - lam_i))
    den = 2 * R**3 * (lam_m + 2 * lam_o) * (lam_m + 0.5 * lam_i) - 2 * (R - d)**3 * (lam_o - lam_m) * (lam_i - lam_m)
    return num / den

lam0 = 3e-7
rows = [{"fold": k, "lambda_m (S/m)": lam0 * k, "f": f_eq2(lam0 * k),
         "reduction_vs_IMPC_%": 100 * (1 - f_eq2(lam0 * k) / 1.5),
         "reduction_vs_intact_f_%": 100 * (1 - f_eq2(lam0 * k) / f_eq2(lam0))} for k in (1, 10, 100)]
print(pd.DataFrame(rows).to_string(index=False))

**Result: partially reproduced.** Eq. 2 with the paper's constants gives a reduction of 1.33% for a
10-fold increase, outside 2 +- 0.2%, and 11.87% for a 100-fold increase, within 12 +- 1.2%. The variant
relative to the intact f gives 1.20 and 11.75%. This is a computation from the paper's
equation and does not test the dataset.

## Summary

| Claim | Outcome |
|---|---|
| fig6a-ca-entry-1000-1200 | reproduced |
| fig6a-ca-sustained | not reproduced |
| fig6a-780-not-significant | reproduced |
| fig6b-train-threshold | not reproduced |
| fig6b-650-weak-but-significant | not reproduced |
| fig5b-conductivity-calibration | partially reproduced (no recordings used) |
| 21 claims on potentiometric imaging (Figs. 1 to 5 and 7) | blocked: data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's: the cell regions are found by a segmentation, not drawn by
hand; the region size and the cell-free background region are standard choices; the voltages of the
files are converted to field strengths at 3.9 V/cm per V (M1), an assumption that fits the fields and group
sizes of Fig. 6B, with 3.3 V/cm per V (M2) shown throughout; all cells in every file are used, where the paper
used 14 cells for Fig. 6A and 224 cells, each tested once, for Fig. 6B; the tests the paper does not name are
the claims file's readings. No outcome changes between M1 and M2.

What was found about the dataset: it holds the Fluo-4 imaging of Fig. 6 and none of the potentiometric
imaging; it gives voltages, not field strengths, and an electrode gap of 1.00 mm where the paper gives
1.6 mm; the train session reuses the same fields of view at up to seven voltages (82 distinct cells,
311 exposures), where the paper says each cell was tested once (224 cells); the single-pulse session holds
27 cells where the paper averaged 14; the description mentions only single pulses though 33 of 73 files
are 60-pulse trains; there are no cell regions, so the paper's cell selection cannot be repeated. The Fluo-4
fluorescence of unpulsed cells declines by 12 to 16% over a recording (8.7 and 7.5% before background
subtraction, with the background nearly flat), where the 0 V/cm trace of Fig. 6A rises by about 1% and
the Fig. 6B traces are flat; the decline decides fig6a-ca-sustained. It is steepest in the first recording of
a field of view: in the single-pulse session it slows with each recording, and in the train session it is
11.4% per minute in the first and 2.0 to 5.4% per minute later, without a steady trend. Field strength and
recording order therefore go together in both sessions. A background taken as a low percentile of the
cell-free pixels, less affected by dim cells there, leaves the decline and every outcome unchanged.
